# Lab 6: Statistical Analysis of Palmer Penguins

Run this notebook with an R kernel or in Google Colab using an R runtime. The cells mirror the `.R` deliverable.

In [ ]:
# Lab 6: Statistical Analysis of Palmer Penguins
# Dataset source: https://github.com/allisonhorst/palmerpenguins

needed <- c("ggplot2", "dplyr", "e1071", "car", "effsize")
to_install <- needed[!needed %in% rownames(installed.packages())]
if (length(to_install) > 0) install.packages(to_install, repos = "https://cloud.r-project.org")

library(ggplot2)
library(dplyr)
library(e1071)
library(car)
library(effsize)

dir.create("outputs", showWarnings = FALSE)
dir.create("plots", showWarnings = FALSE)

penguins <- read.csv("data/penguins.csv", na.strings = c("NA", ""))
penguins_clean <- penguins %>% filter(!is.na(body_mass_g), !is.na(flipper_length_mm), !is.na(species))

desc_body <- penguins_clean %>%
  summarise(
    n = n(),
    mean = mean(body_mass_g),
    median = median(body_mass_g),
    minimum = min(body_mass_g),
    maximum = max(body_mass_g),
    variance = var(body_mass_g),
    standard_deviation = sd(body_mass_g),
    q1 = quantile(body_mass_g, 0.25),
    q3 = quantile(body_mass_g, 0.75),
    iqr = IQR(body_mass_g),
    skewness = skewness(body_mass_g),
    kurtosis = kurtosis(body_mass_g)
  )
write.csv(desc_body, "outputs/descriptive_body_mass_overall_from_R.csv", row.names = FALSE)

desc_species <- penguins_clean %>%
  group_by(species) %>%
  summarise(
    n = n(),
    mean = mean(body_mass_g),
    median = median(body_mass_g),
    minimum = min(body_mass_g),
    maximum = max(body_mass_g),
    variance = var(body_mass_g),
    standard_deviation = sd(body_mass_g),
    q1 = quantile(body_mass_g, 0.25),
    q3 = quantile(body_mass_g, 0.75),
    iqr = IQR(body_mass_g),
    skewness = skewness(body_mass_g),
    kurtosis = kurtosis(body_mass_g),
    .groups = "drop"
  )
write.csv(desc_species, "outputs/descriptive_body_mass_by_species_from_R.csv", row.names = FALSE)

png("plots/histogram_body_mass_R.png", width = 1000, height = 700)
print(ggplot(penguins_clean, aes(x = body_mass_g)) +
  geom_histogram(bins = 20, fill = "#68a6a2", color = "white") +
  labs(title = "Histogram of Penguin Body Mass", x = "Body mass (g)", y = "Count") +
  theme_minimal())
dev.off()

png("plots/boxplot_body_mass_by_species_R.png", width = 1000, height = 700)
print(ggplot(penguins_clean, aes(x = species, y = body_mass_g, fill = species)) +
  geom_boxplot() +
  labs(title = "Species-wise Boxplot of Body Mass", x = "Species", y = "Body mass (g)") +
  theme_minimal())
dev.off()

png("plots/density_body_mass_R.png", width = 1000, height = 700)
print(ggplot(penguins_clean, aes(x = body_mass_g, color = species, fill = species)) +
  geom_density(alpha = 0.2) +
  labs(title = "Density Plot of Body Mass", x = "Body mass (g)", y = "Density") +
  theme_minimal())
dev.off()

png("plots/boxplot_body_mass_by_sex_R.png", width = 1000, height = 700)
print(ggplot(filter(penguins_clean, !is.na(sex)), aes(x = sex, y = body_mass_g, fill = sex)) +
  geom_boxplot() +
  labs(title = "Sex-wise Boxplot of Body Mass", x = "Sex", y = "Body mass (g)") +
  theme_minimal())
dev.off()

png("plots/qqplot_body_mass_R.png", width = 1000, height = 700)
qqnorm(penguins_clean$body_mass_g, main = "QQ Plot of Body Mass")
qqline(penguins_clean$body_mass_g, col = "red")
dev.off()

normality_by_species <- penguins_clean %>%
  group_by(species) %>%
  summarise(
    shapiro_w = shapiro.test(body_mass_g)$statistic,
    shapiro_p_value = shapiro.test(body_mass_g)$p.value,
    .groups = "drop"
  )
write.csv(normality_by_species, "outputs/shapiro_body_mass_by_species_from_R.csv", row.names = FALSE)

sex_data <- penguins_clean %>% filter(!is.na(sex))
t_result <- t.test(body_mass_g ~ sex, data = sex_data, var.equal = FALSE)
cohen_result <- cohen.d(body_mass_g ~ sex, data = sex_data)
sink("outputs/hypothesis_test_body_mass_by_sex_from_R.txt")
cat("H0: Mean body mass is equal for male and female penguins.\n")
cat("H1: Mean body mass differs between male and female penguins.\n\n")
print(t_result)
print(cohen_result)
sink()

levene_result <- leveneTest(body_mass_g ~ species, data = penguins_clean)
anova_body <- aov(body_mass_g ~ species, data = penguins_clean)
tukey_body <- TukeyHSD(anova_body)
kruskal_body <- kruskal.test(body_mass_g ~ species, data = penguins_clean)

write.csv(as.data.frame(summary(anova_body)[[1]]), "outputs/anova_body_mass_by_species_from_R.csv")
write.csv(as.data.frame(tukey_body$species), "outputs/tukey_hsd_body_mass_by_species_from_R.csv")
write.csv(as.data.frame(levene_result), "outputs/levene_body_mass_by_species_from_R.csv")
write.csv(data.frame(statistic = kruskal_body$statistic, df = kruskal_body$parameter, p_value = kruskal_body$p.value), "outputs/kruskal_wallis_body_mass_by_species_from_R.csv", row.names = FALSE)

two_way <- aov(body_mass_g ~ species * sex, data = sex_data)
write.csv(as.data.frame(summary(two_way)[[1]]), "outputs/two_way_anova_body_mass_species_sex_from_R.csv")

anova_flipper <- aov(flipper_length_mm ~ species, data = penguins_clean)
write.csv(as.data.frame(summary(anova_flipper)[[1]]), "outputs/anova_flipper_length_by_species_from_R.csv")

png("plots/boxplot_flipper_length_by_species_R.png", width = 1000, height = 700)
print(ggplot(penguins_clean, aes(x = species, y = flipper_length_mm, fill = species)) +
  geom_boxplot() +
  labs(title = "Species-wise Flipper Length Comparison", x = "Species", y = "Flipper length (mm)") +
  theme_minimal())
dev.off()

png("plots/group_comparison_body_mass_R.png", width = 1000, height = 700)
print(ggplot(penguins_clean, aes(x = species, y = body_mass_g, fill = species)) +
  geom_boxplot(alpha = 0.75) +
  stat_summary(fun = mean, geom = "point", shape = 23, size = 4, fill = "white") +
  labs(title = "Group Comparison of Body Mass by Species", x = "Species", y = "Body mass (g)") +
  theme_minimal())
dev.off()


## Submission Notes

- Upload this notebook and export it to PDF from Colab.
- Include `data/penguins.csv`, the `outputs/` CSV files, and the `plots/` images in the Git upload.
